# Neural Network From Scratch

This notebook breaks down the custom NumPy feed-forward neural network powering the application.

## Learning Objectives
- Understand the exact architecture (784 → 128 → 64 → 10).
- Examine how forward propagation is implemented with linear algebra.
- See the actual weights and shapes of the production model.

In [1]:
import sys
import pathlib

# Dynamically resolve project root
PROJECT_ROOT = pathlib.Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    
print(f"Project root added to path: {PROJECT_ROOT}")

Project root added to path: D:\project\DIGIT Recognition


## Architecture Overview

The model relies exclusively on matrix multiplication and activation functions:

    X      = (1, 784)
    W1     = (784, 128)
    b1     = (128)
    Z1     = (1, 128)   where Z1 = X @ W1 + b1
    A1     = ReLU(Z1)

    W2     = (128, 64)
    Z2     = A1 @ W2 + b2
    A2     = ReLU(Z2)

    W3     = (64, 10)
    Z3     = A2 @ W3 + b3
    Output = Softmax(Z3)

Let's load the production model and inspect it.

In [2]:
from backend.app.model.network import NeuralNetwork
from backend.app.model.layers import Dense

# Initialize empty network
model = NeuralNetwork()

# Load pretrained production weights
weights_path = PROJECT_ROOT / "backend" / "weights" / "model.npz"
model.load(weights_path)

print("Model Loaded Successfully!")

Model Loaded Successfully!


## Inspecting the Layers
Let's look at the actual weight matrix shapes inside our loaded model.

In [3]:
for i, layer in enumerate(model.layers):
    if isinstance(layer, Dense):
        print(f"Layer {i} (Dense): Weights = {layer.weights.shape}, Biases = {layer.biases.shape}")
    else:
        print(f"Layer {i} (Activation): {layer.__class__.__name__}")

Layer 0 (Dense): Weights = (784, 128), Biases = (1, 128)
Layer 1 (Activation): ReLU
Layer 2 (Dense): Weights = (128, 64), Biases = (1, 64)
Layer 3 (Activation): ReLU
Layer 4 (Dense): Weights = (64, 10), Biases = (1, 10)
Layer 5 (Activation): Softmax


## Mathematics

**ReLU (Rectified Linear Unit):** `f(x) = max(0, x)`
This introduces non-linearity, allowing the network to learn complex patterns.

**Softmax:** `p_i = exp(z_i) / Σ exp(z_j)`
Converts the final raw scores (logits) into a valid probability distribution (summing to 1.0).

Let's do a manual forward pass!

In [4]:
from backend.app.dataset.mnist_loader import load_mnist
import numpy as np

# Get one image
images, labels = load_mnist(split="test", subset_size=1)
sample_image = images[0:1] # shape (1, 784)

# Manual forward pass step-by-step
output = sample_image

print(f"Input shape: {output.shape}\n")

for i, layer in enumerate(model.layers):
    output = layer.forward(output)
    if isinstance(layer, Dense):
        print(f"After Dense {i} -> shape: {output.shape}")
    else:
        print(f"After {layer.__class__.__name__} -> shape: {output.shape}, min: {output.min():.2f}, max: {output.max():.2f}")

print(f"\nFinal Probabilities: \n{np.round(output, 3)}")
print(f"\nPredicted Class: {np.argmax(output)}, True Class: {labels[0]}")

Input shape: (1, 784)

After Dense 0 -> shape: (1, 128)
After ReLU -> shape: (1, 128), min: 0.00, max: 2.32
After Dense 2 -> shape: (1, 64)
After ReLU -> shape: (1, 64), min: 0.00, max: 5.80
After Dense 4 -> shape: (1, 10)
After Softmax -> shape: (1, 10), min: 0.00, max: 1.00

Final Probabilities: 
[[0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]]

Predicted Class: 7, True Class: 7


## Key Takeaways
- A neural network is fundamentally a sequence of matrix multiplications and non-linear activation functions.
- The shapes of the weight matrices dictate how many features are extracted at each layer.

## Interview Questions
- **Why use ReLU?** ReLU is computationally cheap and avoids the vanishing gradient problem common with Sigmoid/Tanh activations.
- **What is Softmax?** It is a function that scales numbers into probabilities. It exponentiates the inputs (making them positive) and divides by the sum, ensuring all outputs sum to 1.0, which is perfect for mutually exclusive multi-class classification.